# Phase 1 — Baseline MLP for Chest X-Ray Classification

**Learning Goals**
- Build and train an MLP using the Keras Sequential API
- Compare activation functions (ReLU, LeakyReLU, ELU)
- Compare optimizers (SGD, Adam, RMSprop, Nadam)
- Implement a learning-rate schedule

**Tasks to complete:**
1. `TODO 1` in `src/data_loader.py` — `get_data_generators()`
2. `TODO 3` in `src/data_loader.py` — `compute_class_weights()`
3. `TODO 4` in `src/models.py` — `build_baseline_mlp()`
4. `TODO 9` in `src/train.py` — `get_callbacks()`
5. `TODO 10` in `src/train.py` — `get_lr_schedule()`
6. `TODO 11` in `src/train.py` — `train_model()`

Run `pytest tests/test_phase1.py -v` to check your work.

In [39]:
# Standard imports — do not modify
import sys
sys.path.insert(0, '..')   # makes src/ importable from notebooks/

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
print('TensorFlow version:', tf.__version__)

TensorFlow version: 2.21.0


## 1. Dataset Setup

Download the dataset from Kaggle first:
```bash
kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
unzip chest-xray-pneumonia.zip -d ../data/
```
After unzipping, the structure should be:
```
data/chest_xray/train/NORMAL/
data/chest_xray/train/PNEUMONIA/
data/chest_xray/val/
data/chest_xray/test/
```

In [40]:
from src.data_loader import describe_dataset, get_data_generators, compute_class_weights

# Inspect dataset structure
summary = describe_dataset('../data/chest_xray')
for split, counts in summary.items():
    print(f'{split}: {counts}')

train: {'NORMAL': 1073, 'PNEUMONIA': 1841}
val: {'NORMAL': 268, 'PNEUMONIA': 459}
test: {'NORMAL': 234, 'PNEUMONIA': 390}


In [48]:
# TODO: implement get_data_generators() in src/data_loader.py, then run this cell
train_gen, val_gen, test_gen = get_data_generators(
    data_dir='../data/chest_xray',
    img_size=(64, 64),
    batch_size=32,
    augment_train=True,
)
print('Train batches:', len(train_gen))
print('Val   batches:', len(val_gen))
print('Test  batches:', len(test_gen))

Found 2914 images belonging to 2 classes.
Found 624 images belonging to 2 classes.
Found 624 images belonging to 2 classes.
Train batches: 92
Val   batches: 20
Test  batches: 20


In [43]:
# Visualise a batch of training images
# Retry getting a batch in case of corrupted images
imgs, labels = None, None
for _ in range(5):
    try:
        imgs, labels = next(train_gen)
        break
    except Exception as e:
        print(f"Skipping batch due to error: {e}")
        continue

if imgs is not None:
    fig, axes = plt.subplots(2, 4, figsize=(14, 6))
    for ax, img, lbl in zip(axes.flat, imgs, labels):
        ax.imshow(img.squeeze(), cmap='gray')
        ax.set_title('PNEUMONIA' if lbl == 1 else 'NORMAL')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("Could not load a valid batch after retries")

Skipping batch due to error: cannot identify image file <_io.BytesIO object at 0x00000251C6967D30>
Skipping batch due to error: cannot identify image file <_io.BytesIO object at 0x00000251C6967D30>
Skipping batch due to error: cannot identify image file <_io.BytesIO object at 0x00000251C6967D30>
Skipping batch due to error: cannot identify image file <_io.BytesIO object at 0x00000251C6967D30>
Skipping batch due to error: cannot identify image file <_io.BytesIO object at 0x00000251C6967D30>
Could not load a valid batch after retries


In [44]:
# TODO: implement compute_class_weights() in src/data_loader.py
class_weights = compute_class_weights(train_gen)
print('Class weights:', class_weights)

Class weights: {0: np.float64(1.357875116495806), 1: np.float64(0.7914177077675176)}


## 2. Build and Train the Baseline MLP

Implement `build_baseline_mlp()` in `src/models.py` before running the cells below.

In [45]:
from src.models import build_baseline_mlp

model = build_baseline_mlp(
    input_shape=(64, 64, 3),
    activation='relu',
    optimizer='adam',
    learning_rate=1e-3,
)
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_5 (Flatten)             │ (None, 12288)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 512)            │     6,291,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,456,321 (24.63 MB)

 Trainable params: 6,456,321 (24.63 MB)

 Non-trainable params: 0 (0.00 B)

In [49]:
from src.train import get_callbacks, get_lr_schedule, train_model, plot_history

callbacks = get_callbacks(model_name='baseline_mlp', patience=5)
lr_cb = get_lr_schedule(schedule_type='cosine', initial_lr=1e-3, epochs=20)

history = train_model(
    model,
    train_gen,
    val_gen,
    epochs=20,
    class_weights=class_weights,
    callbacks=callbacks + [lr_cb],
)
plot_history(history, title='Baseline MLP')

UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x00000251C6965170>

## 3. Experiment: Activation Functions

Train three models with different activations and compare validation accuracy.

In [26]:
results = {}
for act in ['relu', 'leaky_relu', 'elu']:
    m = build_baseline_mlp(input_shape=(64, 64, 3), activation=act)
    h = train_model(m, train_gen, val_gen, epochs=10, class_weights=class_weights)
    results[act] = max(h.history['val_accuracy'])
    print(f'{act}: best val_accuracy = {results[act]:.4f}')

print('\nBest activation:', max(results, key=results.get))

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\chest_xray\\train\\PNEUMONIA\\._person1288_virus_2211.jpeg'

## 4. Experiment: Optimizers

Compare SGD, Adam, RMSprop, and Nadam.

In [9]:
opt_results = {}
for opt in ['sgd', 'adam', 'rmsprop', 'nadam']:
    m = build_baseline_mlp(input_shape=(64, 64, 3), optimizer=opt)
    h = train_model(m, train_gen, val_gen, epochs=10, class_weights=class_weights)
    opt_results[opt] = max(h.history['val_accuracy'])
    print(f'{opt}: best val_accuracy = {opt_results[opt]:.4f}')

print('\nBest optimizer:', max(opt_results, key=opt_results.get))

UnidentifiedImageError: cannot identify image file <_io.BytesIO object at 0x00000251BDBCA520>

In [47]:
from pathlib import Path

data_root = Path("../data/chest_xray")

for split in ["train", "val", "test"]:
    split_dir = data_root / split
    for class_name in ["NORMAL", "PNEUMONIA"]:
        count = len(list((split_dir / class_name).glob("*.*")))
        print(f"{split}/{class_name}: {count} files")

train/NORMAL: 1074 files
train/PNEUMONIA: 1841 files
val/NORMAL: 268 files
val/PNEUMONIA: 460 files
test/NORMAL: 234 files
test/PNEUMONIA: 390 files


## 5. Reflection Questions

Answer in the cells below (Markdown):

1. Why is class weighting necessary for this dataset?
2. How does the choice of activation function affect gradient flow?
3. What are the trade-offs between SGD and Adam?
4. What is the effect of learning-rate scheduling on training stability?

**Your answers here:**

1. 
2. 
3. 
4. 